In [2]:
from slava.models.huggingface import HuggingFaceModel
from slava.modules.data_loader import DataLoader
from slava.modules.model_eval import ModelEval

In [6]:
import anthropic

In [8]:
client = anthropic.Anthropic(
  api_key=CLAUDE_API_KEY,
)

message_batch = client.beta.messages.batches.create(
    requests=[
        {
            "custom_id": "first-prompt-in-my-batch",
            "params": {
                "model": "claude-3-haiku-20240307",
                "max_tokens": 100,
                "messages": [
                    {
                        "role": "user",
                        "content": "Hey Claude, tell me a short fun fact about video games!",
                    }
                ],
            },
        },
        {
            "custom_id": "second-prompt-in-my-batch",
            "params": {
                "model": "claude-3-5-sonnet-20240620",
                "max_tokens": 100,
                "messages": [
                    {
                        "role": "user",
                        "content": "Hey Claude, tell me a short fun fact about bees!",
                    }
                ],
            },
        },
    ]
)
print(message_batch)

BetaMessageBatch(id='msgbatch_01BiY1VYTGErVjYW7abQmtha', cancel_initiated_at=None, created_at=datetime.datetime(2024, 10, 10, 11, 17, 42, 111969, tzinfo=datetime.timezone.utc), ended_at=None, expires_at=datetime.datetime(2024, 10, 11, 11, 17, 42, 111969, tzinfo=datetime.timezone.utc), processing_status='in_progress', request_counts=BetaMessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0), results_url=None, type='message_batch')


In [11]:
client = anthropic.Anthropic(
    api_key=CLAUDE_API_KEY,
)
message = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Привет, Claude"}
    ]
)

In [12]:
print(message.content)

[TextBlock(text='Привет! Рад познакомиться. Чем я могу вам помочь сегодня?', type='text')]


In [20]:
import logging
import anthropic

class ClaudeHandler:
    """Класс для работы с моделью Claude от Anthropic."""

    def __init__(self, api_key: str):
        """
        Инициализирует ClaudeHandler с предоставленным API ключом.

        Args:
            api_key (str): Ваш API ключ для Anthropic.
        """
        self.api_key = api_key
        self.client = anthropic.Anthropic(api_key=api_key)

    def generate_response(self, prompt: str) -> str:
        """Генерирует ответ от Claude на основе заданного промпта.

        Args:
            prompt (str): Промпт для модели.

        Returns:
            str: Ответ модели.
        """
        try:
            message = self.client.messages.create(
                model="claude-3-5-sonnet-20240620",
                max_tokens=150,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            if isinstance(message.content, list):
                response_text = ''.join(
                    block.text for block in message.content if hasattr(block, 'text')
                )
                return response_text.strip()
            else:
                return message.content.strip()
        except Exception as e:
            logging.error(f"Ошибка при генерации ответа: {e}")
            return "Ошибка при генерации ответа."


In [23]:
model_handler = ClaudeHandler(api_key=CLAUDE_API_KEY)
data_loader = DataLoader()
model_eval = ModelEval()

dataset = data_loader.load_data()

model_eval.run_evaluation(dataset, model_handler, num_rows = 2840)

100%|██████████| 2840/2840 [57:15<00:00,  1.21s/it]   
